# 02 - Data Cleaning & Feature Engineering
## RHoMIS Analytics — Section C, Group 7

---

This notebook focuses on refining the raw dataset by selecting critical core columns, converting data types, handling sparsity, standardizing variables (like hectares), fixing severe demographic outliers, and engineering KPIs like Food Security Status and Income Per Capita.

In [ ]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

---
## 1. Load Data & Column Reduction
We drop the 1,400+ highly sparse columns and isolate only our focused domain indicators.

In [ ]:
raw_path = '../data/raw/raw.csv.gz'
df = pd.read_csv(raw_path, low_memory=False)

columns_to_keep = [
    'id_unique', 'country', 'region', 'year', 'count_people', 'respondentsex', 'age_malehead', 'age_femalehead',
    'landcultivated', 'landowned', 'unitland', 'land_tenure', 'local_currency',
    'crop_count', 'crop_name_1', 'crop_yield_1', 'crop_harvest_1',
    'crop_income_per_year_1', 'livestock_sale_income_1', 'offfarm_incomes_any',
    'hfias_1', 'hfias_2', 'hfias_3', 'hfias_4', 'hfias_5', 'hfias_6', 'hfias_7', 'hfias_8', 'hfias_9',
    'crop_consumed_prop_1'
]

df_clean = df[columns_to_keep].copy()
print(f"Reduced shape: {df_clean.shape}")

---
## 2. Demographic Standardization & Outlier Management
We standardize categorical inputs for sex, correct ages entered incorrectly as birth years, and prevent 0-population infinite division errors.

In [ ]:
# Standardize sex
sex_map = {'m': 'Male', 'male': 'Male', 'f': 'Female', 'female': 'Female', 'both': 'Both', 'na': np.nan}
df_clean['respondentsex'] = df_clean['respondentsex'].str.lower().str.strip().map(sex_map).fillna(np.nan)

def fix_age(age):
    if pd.isna(age): return np.nan
    try: age = float(age)
    except: return np.nan
    # Convert birth years into ages
    if 1900 < age < 2025: return 2024 - age
    # Drop impossible ages
    if age > 120 or age <= 0: return np.nan
    return age

df_clean['age_malehead'] = df_clean['age_malehead'].apply(fix_age)
df_clean['age_femalehead'] = df_clean['age_femalehead'].apply(fix_age)

# Ensure count_people is numeric & minimum 1
df_clean['count_people'] = pd.to_numeric(df_clean['count_people'], errors='coerce')
df_clean['count_people'] = df_clean['count_people'].fillna(df_clean['count_people'].median()).clip(lower=1)


---
## 3. Standardizing Hectares (Land Size)
Units are mixed between `hectares`, `acres`, and local variants like `tim` or `feddan`. We standardize everything primarily focusing on `hectare` and `acre` where explicitly defined.

In [ ]:
def standardize_hectares(row):
    val = row.get('landcultivated', np.nan)
    unit = str(row.get('unitland', '')).lower()
    if pd.isna(val) or val == '': return np.nan
    try: val = float(val)
    except ValueError: return np.nan
    
    if 'acre' in unit:
        return val * 0.404686
    return val # Standardize assuming local mapping defaulted to HA

df_clean['land_cultivated_ha'] = df_clean.apply(standardize_hectares, axis=1)
df_clean['land_cultivated_ha'] = df_clean['land_cultivated_ha'].clip(upper=1500) # Cap extreme outliers

df_clean['farm_size_bucket'] = pd.cut(
    df_clean['land_cultivated_ha'],
    bins=[-np.inf, 2, 10, np.inf], 
    labels=['Smallholder (<=2ha)', 'Medium (2-10ha)', 'Large (>10ha)']
)


---
## 4. Engineering Income Variables

In [ ]:
for col in ['crop_income_per_year_1', 'livestock_sale_income_1']:
    df_clean[col] = pd.to_numeric(df_clean[col], errors='coerce').fillna(0)

df_clean['total_annual_income'] = df_clean['crop_income_per_year_1'] + df_clean['livestock_sale_income_1']
df_clean['income_per_capita'] = df_clean['total_annual_income'] / df_clean['count_people']
print(f"Average Income Per Capita: {df_clean['income_per_capita'].mean():.2f}")

---
## 5. Modeling Food Security (HFIAS Score)
HFIAS answers represent frequency: `never`, `rarely`, `sometimes`, `often`. We map these linearly to a 0-3 severity weight. Max score is 27.

In [ ]:
hfias_map = {
    'never': 0, 
    'fewpermonth': 1, 'monthly': 1, 
    'fewperweek': 2, 'weekly': 2, 
    'daily': 3
}

hfias_cols = [f'hfias_{i}' for i in range(1, 10)]
score_cols = []
for c in hfias_cols:
    score_cols.append(f'{c}_score')
    df_clean[f'{c}_score'] = df_clean[c].str.lower().map(hfias_map)

# Only sum rows that have at least one valid HFIAS answer
df_clean['hfias_total_score'] = df_clean[score_cols].sum(axis=1, min_count=1)

def map_food_security(val):
    if pd.isna(val): return np.nan
    if val <= 5: return 'Food Secure'
    if val <= 10: return 'Mildly Insecure'
    if val <= 17: return 'Moderately Insecure'
    return 'Severely Insecure'

df_clean['food_security_status'] = df_clean['hfias_total_score'].apply(map_food_security)


---
## 6. Export Processed Data
We drop intermediate calculation columns and push to the clean processing directory for Tableau and further modeling.

In [ ]:
df_final = df_clean.drop(columns=score_cols + hfias_cols + ['unitland', 'landcultivated'])

processed_path = '../data/processed/cleaned_rhomis_data.csv'
df_final.to_csv(processed_path, index=False)
print(f"Data saved successfully to {processed_path}")
print(f"Final Schema: {df_final.shape}")